# R2-Dreamer / DreamerV3 Foundation

This is the single training notebook for the R2-Dreamer track. It trains a DreamerV3 / R2-Dreamer world model on a DMC Walker `walk -> run` transfer and runs the source / fine-tune / scratch comparison, writing model-only interval checkpoints to Google Drive.

It consolidates the two earlier training notebooks (the A100 `dmc_vision` pillar and the T4 `dmc_proprio` pillar) into one preset-driven notebook. It is self-contained: it mounts Drive, defines every path, ensures this repository exists under `/content/World_Models_LAS`, selects a run preset, prepares the external `NM512/r2dreamer` checkout, applies the local patches, and runs a smoke test followed by the long source / fine-tune / scratch runs through a live monitoring panel.

**Assumes an A100.** The default `a100_vision` preset uses `dmc_vision`, `size25M`, `model.rep_loss=r2dreamer`, real MuJoCo image rendering, and an 800K / 400K step budget. To reproduce the smaller T4-safe run instead, set `R2_PRESET=t4_proprio` before running the preset cell; that switches to `dmc_proprio`, `size12M`, and disables image rendering.

Use a Colab Python 3.11 runtime: upstream `r2dreamer` requires Python `>=3.11,<3.12`. The long-running cells are separated and meant to be run manually after the smoke test passes.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

# Mount Drive. Colab requires explicit user approval for this step.
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/wm_poc")
for subdir in [
    "data",
    "checkpoints",
    "logs/r2dreamer",
    "figures/r2dreamer",
    "tensorboard/r2dreamer",
    "videos/r2dreamer",
    "reports",
]:
    (DRIVE_ROOT / subdir).mkdir(parents=True, exist_ok=True)

runtime_env = {
    "WM_POC_REPO": "/content/World_Models_LAS",
    "WM_POC_DRIVE_ROOT": str(DRIVE_ROOT),
    "WM_POC_DATA_DIR": str(DRIVE_ROOT / "data"),
    "WM_POC_CKPT_DIR": str(DRIVE_ROOT / "checkpoints"),
    "WM_POC_LOG_DIR": str(DRIVE_ROOT / "logs"),
    "WM_POC_FIG_DIR": str(DRIVE_ROOT / "figures"),
    "WM_POC_FIGURE_DIR": str(DRIVE_ROOT / "figures"),
    "WM_POC_TB_DIR": str(DRIVE_ROOT / "tensorboard"),
    "WM_POC_VIDEO_DIR": str(DRIVE_ROOT / "videos"),
    "WM_POC_EXTERNAL_REPOS": "/content/external_repos",
    "WM_POC_REPORT_DIR": str(DRIVE_ROOT / "reports"),
    "R2DREAMER_REPO": "/content/external_repos/r2dreamer",
    "R2_LOG_ROOT": str(DRIVE_ROOT / "logs" / "r2dreamer"),
    "R2_FIGURE_DIR": str(DRIVE_ROOT / "figures" / "r2dreamer"),
}
os.environ.update(runtime_env)

Path(os.environ["WM_POC_EXTERNAL_REPOS"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["R2_LOG_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["R2_FIGURE_DIR"]).mkdir(parents=True, exist_ok=True)

for key in [
    "WM_POC_REPO",
    "WM_POC_DRIVE_ROOT",
    "WM_POC_EXTERNAL_REPOS",
    "R2DREAMER_REPO",
    "R2_LOG_ROOT",
    "R2_FIGURE_DIR",
]:
    print(key, "=", os.environ[key])

In [ ]:
%%bash
set -e
nvidia-smi || true
python - <<'PY'
import sys
try:
    import torch
except Exception as exc:
    print("torch import failed:", repr(exc))
else:
    print("python:", sys.version)
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    print("cuda device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
    print("cuda capability:", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None)
PY

## Ensure this repository exists in the Colab runtime

The notebook file can be visible in VS Code while the Colab VM still lacks the repository files under `/content`. This cell makes the runtime copy explicit.

In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO_URL = "https://github.com/Thomas-Georges/World_Models_LAS.git"
REPO_DIR = Path("/content/World_Models_LAS")


def run_git(args, *, check=True):
    print("$", " ".join(map(str, args)))
    return subprocess.run(args, check=check)


def sync_repo():
    if not (REPO_DIR / ".git").is_dir():
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)
        run_git(["git", "clone", REPO_URL, str(REPO_DIR)])
    else:
        run_git(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])
        run_git(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"])
        run_git(["git", "-C", str(REPO_DIR), "checkout", "main"])
        run_git(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"])


sync_repo()


command = ["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"]
print("$", " ".join(command))
head = subprocess.check_output(command, text=True).strip()
print("Repository HEAD:", head)

## Select run preset

Choose the walk -> run transfer preset. The default is the A100 `dmc_vision` pillar (`size25M`, real image rendering). To run the T4-safe `dmc_proprio` pillar instead, set `R2_PRESET=t4_proprio` in the environment (or edit `PRESET` below) before running this cell.

This cell sets `R2_CONFIG`, `R2_LOG_ROOT`, `R2_FIGURE_DIR`, the MuJoCo rendering backend, and the upstream setup flags so the remaining cells adapt to the selected preset automatically.

In [ ]:
from pathlib import Path
import ctypes.util
import os
import subprocess

# Select which walk -> run transfer preset to train.
#   "a100_vision" -> dmc_vision, size25M, r2dreamer, real MuJoCo rendering (default; assumes an A100).
#   "t4_proprio"  -> dmc_proprio, size12M, r2dreamer, no image rendering (Colab T4-safe fallback).
PRESET = os.environ.get("R2_PRESET", "a100_vision").strip().lower()

PRESETS = {
    "a100_vision": {
        "config": "configs/r2dreamer/three_way_walker_walk_to_run_a100_r2_vision25m.yaml",
        "run_name": "walker_walk_to_run_a100_r2_vision25m_seed0",
        "render_images": True,
    },
    "t4_proprio": {
        "config": "configs/r2dreamer/three_way_walker_walk_to_run_t4_r2_proprio.yaml",
        "run_name": "walker_walk_to_run_t4_r2_proprio_12m_seed0",
        "render_images": False,
    },
}
if PRESET not in PRESETS:
    raise ValueError(f"Unknown R2_PRESET={PRESET!r}; choose one of: {', '.join(PRESETS)}")
preset = PRESETS[PRESET]

DRIVE_ROOT = Path(os.environ.get("WM_POC_DRIVE_ROOT", "/content/drive/MyDrive/wm_poc"))
RUN_NAME = preset["run_name"]
os.environ["R2_PRESET"] = PRESET
os.environ["R2_CONFIG"] = preset["config"]
os.environ["R2_LOG_ROOT"] = str(DRIVE_ROOT / "logs" / "r2dreamer" / RUN_NAME)
os.environ["R2_FIGURE_DIR"] = str(DRIVE_ROOT / "figures" / "r2dreamer" / RUN_NAME)

if preset["render_images"]:
    # Vision runs need real RGB frames. OSMesa is the most stable headless backend on Colab.
    os.environ["WM_POC_DMC_DISABLE_IMAGE_RENDER"] = "false"
    requested_backend = os.environ.get("R2_MUJOCO_GL", "osmesa").strip().lower()
    if requested_backend == "egl" and os.environ.get("R2_ALLOW_EGL") != "1":
        print("Overriding R2_MUJOCO_GL=egl to osmesa for Colab stability (set R2_ALLOW_EGL=1 to keep egl).")
        requested_backend = "osmesa"
    os.environ["R2_MUJOCO_GL"] = requested_backend
    os.environ["MUJOCO_GL"] = requested_backend
    if requested_backend in {"egl", "osmesa"}:
        os.environ["PYOPENGL_PLATFORM"] = requested_backend
    else:
        os.environ.pop("PYOPENGL_PLATFORM", None)
    os.environ["R2_MUJOCO_EGL_DEVICE_ID"] = "0"
    if requested_backend == "osmesa" and ctypes.util.find_library("OSMesa") is None:
        print("Installing OSMesa system packages for headless MuJoCo rendering...")
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "libosmesa6", "libgl1-mesa-glx", "libglfw3"],
            check=True,
        )
else:
    # Proprio runs do not need image rendering; keep Colab startup small.
    os.environ["WM_POC_DMC_DISABLE_IMAGE_RENDER"] = "true"

Path(os.environ["R2_LOG_ROOT"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["R2_FIGURE_DIR"]).mkdir(parents=True, exist_ok=True)

print("PRESET =", PRESET)
print("R2_CONFIG =", os.environ["R2_CONFIG"])
print("R2_LOG_ROOT =", os.environ["R2_LOG_ROOT"])
print("R2_FIGURE_DIR =", os.environ["R2_FIGURE_DIR"])
print("WM_POC_DMC_DISABLE_IMAGE_RENDER =", os.environ["WM_POC_DMC_DISABLE_IMAGE_RENDER"])
print("MuJoCo backend =", os.environ.get("MUJOCO_GL", "(image rendering disabled)"))


## Clone and install r2dreamer

Use a Colab Python 3.11 runtime for training. Upstream `r2dreamer` requires Python `>=3.11,<3.12`, so the install keeps the setup version guard enabled.

The patch cell below adds checkpoint loading/saving support, interval checkpoints, progress heartbeats, live log monitoring, and Colab runtime guards.

In [ ]:
%%bash
set -e
cd /content/World_Models_LAS
bash scripts/r2dreamer/setup_r2dreamer.sh --extras dmc --target-dir /content/external_repos/r2dreamer


In [ ]:
%%bash
set -e
cd /content/World_Models_LAS

python scripts/r2dreamer/patch_checkpoint_loading.py   --r2-repo /content/external_repos/r2dreamer

python scripts/r2dreamer/verify_r2dreamer_patch.py   --r2-repo /content/external_repos/r2dreamer

## Tiny smoke test

This is still a real training command, but with a tiny budget. It uses the selected preset's config, keeps image rendering on for `dmc_vision` (off for `dmc_proprio`), disables `torch.compile`, and updates a live Python display panel from the run log. Run it after setup succeeds and before the long runs.

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path("/content/World_Models_LAS")
os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

os.environ["HYDRA_FULL_ERROR"] = "1"
os.environ["R2_SMOKE_PROGRESS_EVERY"] = os.environ.get("R2_SMOKE_PROGRESS_EVERY", "10")

from wm_poc.r2dreamer.notebook_monitor import run_r2_with_live_display

run_r2_with_live_display("smoke", monitor_interval=5)


## Long-running source run

Run manually after the smoke test passes. Full runs save model-only interval checkpoints at eval boundaries under each run directory, for example `source_base/checkpoints/step_000400000.pt`. Retention is controlled by `R2_CHECKPOINT_KEEP`.

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path("/content/World_Models_LAS")
os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

os.environ["HYDRA_FULL_ERROR"] = "1"

from wm_poc.r2dreamer.notebook_monitor import run_r2_with_live_display

run_r2_with_live_display("source_base")


## Long-running target fine-tune run

Requires `$R2_LOG_ROOT/source_base/latest.pt`.

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path("/content/World_Models_LAS")
os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

os.environ["HYDRA_FULL_ERROR"] = "1"

from wm_poc.r2dreamer.notebook_monitor import run_r2_with_live_display

run_r2_with_live_display("target_finetune")


## Long-running target scratch run

Use the same target budget as the fine-tune run.

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path("/content/World_Models_LAS")
os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

os.environ["HYDRA_FULL_ERROR"] = "1"

from wm_poc.r2dreamer.notebook_monitor import run_r2_with_live_display

run_r2_with_live_display("target_scratch")


## Summaries and fine-tune vs scratch plot

After the runs complete, write the per-run summary CSV and the fine-tune vs scratch evaluation figure. Use `06_r2dreamer_results.ipynb` for the full results, rollout videos, and latent analysis.

In [ ]:
%%bash
set -e
cd /content/World_Models_LAS

echo "Using R2_LOG_ROOT=${R2_LOG_ROOT}"
echo "Using R2_FIGURE_DIR=${R2_FIGURE_DIR}"
python scripts/r2dreamer/summarize_runs.py   --run-root "$R2_LOG_ROOT"   --out "$R2_LOG_ROOT/summary.csv"

python scripts/r2dreamer/plot_finetune_vs_scratch.py   --finetune "$R2_LOG_ROOT/target_finetune/metrics.jsonl"   --scratch "$R2_LOG_ROOT/target_scratch/metrics.jsonl"   --out "$R2_FIGURE_DIR/finetune_vs_scratch.png"
